# 01. VAE: от вероятностной модели к обучению

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

Выведем ELBO и diagonal Gaussian KL, свяжем непрерывную плотность с конечным тензором и проверим reparameterization градиентами.

[Порядок и источники](../README.md) · [Связь с MIT Lab 3](../MIT_LAB3_BRIDGE.md) · [Все компоненты и методы](../SOURCE_MAP.md) · [Результаты](../reports/VALIDATION.md).

Работаем по циклу **формула → прогноз → код → измерение → вывод**. Все ячейки по умолчанию используют CPU; никакие веса, датасеты или ComfyUI jobs не скачиваются/запускаются. Исходные алгоритмы и учебные адаптеры явно различаются. Перед запуском численного опыта сформулируйте ожидаемый результат.


**MIT параллельно:** Lab 3 Q4.7 (cells 68–69), `VAE.forward/compute_loss`; notes §6.2 pp.46–52, Appendix D pp.77–82. После KL и reparameterization допишите свою версию compute_loss; сверяйте reduction и смысл x_logvar. [Задание и checklist](../MIT_LAB3_BRIDGE.md).


In [ ]:
from pathlib import Path
import sys, json, math
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
candidates=[Path.cwd(),*Path.cwd().parents]
COURSE=next((p/'vae' for p in candidates if (p/'vae/labkit').is_dir()),None)
assert COURSE is not None, 'Clone the repository first; see the Colab instructions in the course README. No weights are needed.'
sys.path.insert(0,str(COURSE))
from labkit.vae_lab import (NATIVE,TinyCodec,DiagonalPosterior,oracle_codec,plan,
    axis_coverage,anchored_starts,synthetic_video,errors,measure,show_source,
    source_hash,save_metrics,figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
LAB='01_elbo_and_probabilistic_codec';metrics={}
print('Pinned source:',source_hash())


## 1. Почему нельзя просто интегрировать decoder

Порождающая модель: сначала z из prior, затем x из decoder likelihood. Encoder q — обучаемое приближение недоступного posterior.

$$
p_\theta(x)=\int p_\theta(x\mid z)p(z)\,dz,
\qquad q_\phi(z\mid x)\approx p_\theta(z\mid x).
$$

Домножим подынтегральную функцию на q/q (там, где q положительна) и применим вогнутость log:

$$
\log p_\theta(x)=\log\mathbb{E}_{q_\phi}\left[\frac{p_\theta(x\mid z)p(z)}{q_\phi(z\mid x)}\right]
\geq \mathbb{E}_{q_\phi}[\log p_\theta(x\mid z)]-D_{\mathrm{KL}}(q_\phi\Vert p)=\mathcal{L}.
$$

Точная разница:

$$
\log p_\theta(x)-\mathcal{L}=D_{\mathrm{KL}}(q_\phi(z\mid x)\Vert p_\theta(z\mid x))\geq0.
$$

Чем точнее posterior, тем теснее bound. Для scalar примера p(z)=N(0,1), p(x|z)=N(z,s²) marginal известна: N(0,1+s²). Проверим bound и разрыв Monte Carlo, а не на словах.


In [ ]:
torch.manual_seed(1);dtype=torch.float64
x=torch.tensor(.7,dtype=dtype);s=torch.tensor(.4,dtype=dtype)
q=torch.distributions.Normal(torch.tensor(.2,dtype=dtype),torch.tensor(.6,dtype=dtype))
prior=torch.distributions.Normal(torch.tensor(0.,dtype=dtype),torch.tensor(1.,dtype=dtype))
z=q.sample((200000,))
term=torch.distributions.Normal(z,s).log_prob(x)+prior.log_prob(z)-q.log_prob(z)
elbo=float(term.mean());logp=float(torch.distributions.Normal(0.,torch.sqrt(1+s*s)).log_prob(x))
se=float(term.std()/math.sqrt(len(term)))
assert elbo < logp+5*se
metrics['elbo']={'MC':elbo,'standard_error':se,'exact_log_marginal':logp,'gap':logp-elbo}
print(metrics['elbo'])


## 2. Gaussian posterior: вывод KL по координатам

Для каждого скаляра q=N(μ,σ²), p=N(0,1). Разность log densities:

$$
\log q-\log p=-\log\sigma-\frac{(z-\mu)^2}{2\sigma^2}+\frac{z^2}{2}.
$$

Под q имеем E[(z−μ)²]=σ² и E[z²]=μ²+σ². Поэтому:

$$
D_{\mathrm{KL}}(q\Vert p)=\tfrac12(\mu^2+\sigma^2-1-\log\sigma^2).
$$

Для диагональной ковариации многомерная log density — сумма, значит KL суммируется по C,F,H,W отдельно для каждого B:

$$
\ell=\log\sigma^2,\quad \sigma=\exp(\ell/2),\quad
D_{\mathrm{KL}}=\tfrac12\sum_i(\mu_i^2+\exp(\ell_i)-1-\ell_i).
$$

Проверьте: std не равна exp(logvar); mean reduction и sum reduction дают разные масштабы β. Код source clamp logvar ограничивает численную динамику; это не часть исходного вывода Gaussian KL.


In [ ]:
moments=torch.randn(2,32,3,4,4)*.2
posterior=DiagonalPosterior(moments)
q=torch.distributions.Normal(posterior.mean,posterior.std)
p=torch.distributions.Normal(torch.zeros_like(posterior.mean),torch.ones_like(posterior.std))
expected=torch.distributions.kl_divergence(q,p).flatten(1).sum(1)
assert torch.allclose(posterior.kl(),expected,atol=1e-4)
metrics['kl_max_error']=float((posterior.kl()-expected).abs().max())
print('moments / mean / KL:',moments.shape,posterior.mean.shape,posterior.kl())


## 3. Как gradient проходит через sample

Замена переменной переносит случайность в независимый ε:

$$
z=\mu_\phi(x)+\exp(\ell_\phi(x)/2)\odot\epsilon,\qquad \epsilon\sim N(0,I).
$$

Для одного фиксированного ε:

$$
\frac{\partial z}{\partial\mu}=1,\qquad
\frac{\partial z}{\partial\ell}=\tfrac12\exp(\ell/2)\epsilon.
$$

Так pathwise estimator дифференцирует deterministic graph. Сравним autograd с finite difference при **том же ε**, иначе шум замаскирует derivative. При mode z=μ encoder uncertainty не используется; это полезный deterministic baseline реконструкции.


In [ ]:
mu=torch.tensor(.3,dtype=torch.float64,requires_grad=True)
lv=torch.tensor(-.7,dtype=torch.float64,requires_grad=True);eps=torch.tensor(.8,dtype=torch.float64)
def loss(m,l):return (m+torch.exp(.5*l)*eps-1).square()
g=torch.autograd.grad(loss(mu,lv),(mu,lv));h=1e-5
fd=[(loss(mu+h,lv)-loss(mu-h,lv))/(2*h),(loss(mu,lv+h)-loss(mu,lv-h))/(2*h)]
assert all(torch.allclose(a,b,atol=1e-8) for a,b in zip(g,fd))
metrics['gradient_max_error']=max(float((a-b).abs().detach()) for a,b in zip(g,fd))
print('autograd:',[float(v) for v in g],'finite difference:',[float(v.detach()) for v in fd])


## 4. От likelihood к loss HunyuanVideo

Независимая Gaussian likelihood с фиксированным σₓ даёт squared reconstruction:

$$
-\log p_\theta(x\mid z)=\frac{1}{2\sigma_x^2}\sum_i(x_i-\hat x_i)^2+\mathrm{const}.
$$

Для Laplace likelihood с scale b получается L1/b плюс константа. В отчёте HunyuanVideo v6 Eq. 1 используется:

$$
L=L_1+0.1L_{\mathrm{LPIPS}}+0.05L_{\mathrm{adv}}+10^{-6}L_{\mathrm{KL}}.
$$

Perceptual и adversarial terms расширяют цель за пределы простого ELBO. По одним коэффициентам нельзя воспроизвести весь training recipe: нужны reductions, discriminator, данные, optimizer. При обучении DiT pretrained VAE обычно frozen, его KL loss повторно не оптимизируется.

Обучим крошечный **linear VAE на 2D данных** с MSE+βKL: это упражнение на механизм, не реплика Hunyuan training. Данные x — конечномерный вектор вместо видеотензора; сумма формально та же.


In [ ]:
torch.manual_seed(6)
data=torch.stack((torch.linspace(-1,1,128),torch.linspace(-1,1,128)**2),1)
enc=nn.Linear(2,4);dec=nn.Linear(2,2);opt=torch.optim.Adam([*enc.parameters(),*dec.parameters()],lr=.025)
curve=[]
for step in range(300):
    mu,lv=enc(data).chunk(2,-1);lv=lv.clamp(-8,4)
    z=mu+torch.exp(.5*lv)*torch.randn_like(mu)
    rec=F.mse_loss(dec(z),data)
    kl=.5*(mu.square()+lv.exp()-1-lv).sum(-1).mean()
    loss=rec+.01*kl;opt.zero_grad();loss.backward();opt.step()
    curve.append(float(loss.detach()))
metrics['linear_vae']={'first_30_mean':float(np.mean(curve[:30])),'last_30_mean':float(np.mean(curve[-30:])),'beta':.01}
assert np.mean(curve[-30:])<np.mean(curve[:30])
fig,ax=plt.subplots(figsize=(7,3));ax.plot(curve);ax.set(xlabel='Optimizer step',ylabel='MSE + 0.01 KL',title='Educational 2D linear VAE')
fig.tight_layout();fig.savefig(figure_path(LAB,'training'));plt.show()


## 5. Возвращаемся в MIT Q4.7: reconstruction с learned variance

В MIT Lab 3 `decode` выдаёт x_mean и x_logvar. Условная likelihood по каждому pixel — N(m_i(z),exp(r_i(z))). Тогда с учётом normalizing constant:

$$
-\log p(x\mid z)=\frac12\sum_i\left[\log(2\pi)+r_i+(x_i-m_i)^2\exp(-r_i)\right].
$$

Сначала суммируем по C/H/W (или C/F/H/W для видео), затем усредняем по batch. Это NLL на пример. KL также sum по latent dimensions, mean по batch:

$$
L=\mathbb{E}_{B}[\mathrm{NLL}(x\mid z)+\beta D_{\mathrm{KL}}(q(z\mid x)\Vert p(z))].
$$

MSE эквивалентна Gaussian NLL лишь при фиксированной variance и согласованном масштабе. Без члена r_i learned variance могла бы просто расти, уменьшая weighted error; normalizer штрафует это. x_logvar decoder — uncertainty likelihood, z_logvar encoder — uncertainty posterior. В HunyuanVideo применяется другая reconstruction objective (Eq. 1); одинаковой learned output variance требовать нельзя.

Теперь допишите `VAE.compute_loss` в **своей копии MIT Lab 3, cell 69**. Ниже проверка формулы, не замена canonical задания готовыми ответами.


In [ ]:
torch.manual_seed(12)
x_true=torch.randn(3,1,8,8,dtype=torch.float64)
x_mean=torch.randn_like(x_true);x_logvar=torch.randn_like(x_true)*.2
per_pixel=.5*(math.log(2*math.pi)+x_logvar+(x_true-x_mean).square()*torch.exp(-x_logvar))
nll=per_pixel.flatten(1).sum(1)
reference=-torch.distributions.Normal(x_mean,torch.exp(.5*x_logvar)).log_prob(x_true).flatten(1).sum(1)
assert torch.allclose(nll,reference,atol=1e-10)
metrics['MIT_Q4_7_NLL_max_error']=float((nll-reference).abs().max())
print('Gaussian NLL per example:',nll.tolist())


## Упражнения и критерий перехода

1. Выведите KL для prior N(m₀,s₀²), затем проверьте через torch.distributions.
2. Увеличьте β в 10 раз: сравните reconstruction, KL и latent variance на held-out точках.
3. Почему качество decoder(mode) не равно likelihood estimate? Укажите MC uncertainty. Переход: самостоятельно получите знак и все члены KL без запоминания.

Сначала запишите прогноз, затем меняйте один фактор. Разделяйте ошибку аппроксимации, ошибку интерфейса/формы и аппаратное ограничение. Сохранённые метрики относятся к этому запуску.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Источники

[HunyuanVideo, v6, §4.1 / Fig. 6 / Eq. 1, стр. 5–6](https://arxiv.org/pdf/2412.03603v6) · [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114) · [Официальный HunyuanVideo](https://github.com/Tencent-Hunyuan/HunyuanVideo) · [Закреплённый исследуемый исходник](../../kandinsky-6/reference/kandinsky-6/kandinsky/core/components/hunyuan_vae.py) · [Lecture 4 / исходный MIT Lab 3](../../lecture_04/README.md).

Математические выводы и малые опыты — авторские пояснения. Числа tiny-модели не являются оценкой pretrained HunyuanVideo/Kandinsky. Подробная provenance и ограничения — в SOURCE_MAP.
